In [ ]:
%run _bootstrap_dev.ipynb

# Lazy Portfolio Analyst
Flusso: frequency selection → posizionamento rispetto alla frontiera
efficiente (diagnostico) → backtest PTF → validazione statistica.

La frontiera efficiente è uno strumento diagnostico: i suoi pesi
"ottimali" sono stimati in-sample sull'intero storico e non vanno
adottati come allocazione — sono affetti da overfitting strutturale
(si veda §5).

## §1 — Configurazione

In [ ]:
# Portafoglio da analizzare (definiti in l_portfolios.py)
# my_portfolio       = greta_base_spy_portfolio_etf_ita
# my_portfolio       = greta_alt_emdiv_test
# my_portfolio       = lazy_greta_base_spy
# my_portfolio       = lazy_balanced_60_20_20
# my_portfolio_title = 'Lazy Balanced 60_40'
# my_portfolio_title = "Elektra Balanced "
# my_portfolio_title = "Gretchen S&P 500 Core-Engine - Fiduciaria"
# my_portfolio_title = "Gretchen S&P 500 Core-Engine - Alternative Test"
# my_portfolio_title = "Gretchen S&P 500 Base SPY"

# portfolio       = sandbox_xx
# portfolio       = sandbox_dws_lc
portfolio       = sandbox_proposed_100k
portfolio_title = portfolio['Title']
tickers         = list(portfolio['tickers'].keys())
weights         = list(portfolio['tickers'].values())
benchmark       = portfolio['benchmark']

# Parametri analisi
start_date = '2016-01-01'   # inizio backtest
end_date   = None           # None = oggi
init_cash  = 100_000
fees       = 0.001
years      = 10             # finestra frontiera efficiente (anni) — usato da
                             # efficient_frontier_pypfopt e lazy_stability_weights
                             # per decidere QUANTO storico richiedere.
min_years  = 1               # storico minimo comune richiesto (in anni), passato a
                              # run_bh_backtest (e a cascata a Block B) — soglia sotto
                              # la quale il backtest si rifiuta di procedere.
                              #
                              # NOTA: years e min_years non sono collegati nel codice,
                              # ma vanno tenuti coerenti con lo storico REALE disponibile
                              # per i ticker più corti del portfolio. Se un ticker ha
                              # es. 1.3 anni di dati, sia years sia min_years vanno
                              # abbassati di conseguenza — altrimenti otterrai lo stesso
                              # errore "storico insufficiente" da funzioni diverse
                              # (efficient_frontier_pypfopt/lazy_stability_weights
                              # falliscono su years troppo alto; run_bh_backtest/Block B
                              # falliscono su min_years troppo alto), anche se il
                              # messaggio diagnostico è diverso caso per caso.
print(f"Portafoglio: {BOLD}{portfolio_title}{RESET}")
print(f"Tickers: {tickers}")
print(f"Weights: {weights}")
print(f"Benchmark: {benchmark}")


## §2 — Frequency Selection
Testa W/M/Q/Y/None e seleziona la frequenza con Sharpe massimo.
Cambia `freq_selection_metric` per usare 'cagr' | 'total_return' | 'max_dd'.

In [ ]:
freq_df, best_freq = compare_rebalance_frequencies(
    portfolio, start_date, end_date, init_cash, fees, min_years=min_years
)

## §3 — Posizionamento rispetto alla frontiera efficiente (diagnostico)

Mostra dove si colloca il PTF proposto rispetto allo spazio
rischio/rendimento stimato storicamente. Non è un suggerimento di
allocazione: i punti della frontiera (Min Vol, Max Sharpe, Max Return)
sono ottimizzati in-sample e quindi inevitabilmente overfit — si veda
la nota in §5.

**Legenda colonne:**
- `Return` / `Volatility` / `Sharpe` — valori teorici stimati da
  media storica e covarianza (PyPortfolioOpt, in-sample)
- `Real Return` / `Real Volatility` / `Real Sharpe` — valori
  effettivi misurati sul backtest reale nel periodo selezionato

In [ ]:

fig_frontier, df_special = efficient_frontier_pypfopt(
    tickers=tickers,
    years=years,
    weights=weights,
    n_points=80,
    weight_bounds=(0, 1),
    show_plot=True,
    interactive=True,
    print_weights=True,
)

if df_special is not None and not df_special.empty:    # Pesi ottimali Max Sharpe
    _metric_cols = {'Return', 'Volatility', 'Sharpe',
                    'Real Return', 'Real Volatility', 'Real Sharpe'}
    _weight_cols = [c for c in df_special.columns if c not in _metric_cols]
    optimal_weights = df_special.loc['Max Sharpe', _weight_cols].to_dict()
    
    print("\nPosizionamento PTF proposto vs frontiera (diagnostico):")
    print(f"  My Portfolio  — Sharpe: {df_special.loc['My Portfolio','Sharpe']:.2f}"
          f"  Return: {df_special.loc['My Portfolio','Return']:.2f}%"
          f"  Vol: {df_special.loc['My Portfolio','Volatility']:.2f}%")
    print(f"  Max Sharpe pt — Sharpe: {df_special.loc['Max Sharpe','Sharpe']:.2f}"
          f"  Return: {df_special.loc['Max Sharpe','Return']:.2f}%"
          f"  Vol: {df_special.loc['Max Sharpe','Volatility']:.2f}%")
    print("  ⚠️  Il punto Max Sharpe è in-sample — vedi §5 prima di interpretarlo")
    # optimal_weights resta calcolato per riferimento (usato in §5 nota), non come allocazione consigliata

## §4 — Backtest PTF proposto
Analisi completa con la frequenza ottimale selezionata in §2.

In [ ]:
benchmark_data = download_data(benchmark, start_date, end_date)

pf_proposed = run_bh_backtest(portfolio, start_date, end_date,
                               init_cash, fees, best_freq, min_years=min_years)

if pf_proposed is None:
    print("⚠️  Impossibile generare il report — storico insufficiente per questo portfolio.")
else:
    report_portfolio_title = f"{portfolio_title} - Frequenza {best_freq}"
    out = generate_lazy_portfolio_performance(
        pf=pf_proposed,
        portfolio_title=report_portfolio_title,
        benchmark=benchmark,
        benchmark_data=benchmark_data,
        show_report=True,
        show_plots=True,
        alpha_analysis=True,
    )

## §5 — Perché non adottiamo i pesi "ottimali" della frontiera

Il punto Max Sharpe calcolato in §3 massimizza lo Sharpe ratio
**sullo stesso campione storico** usato per stimarlo (rendimenti
attesi `mu` e covarianza `S` di PyPortfolioOpt). È l'equivalente di
un WFO con un solo fold in-sample e zero fold out-of-sample —
esattamente la trappola che il framework R-portfolio e K-strategy è
costruito per evitare.

In pratica: `mu` (rendimento atteso storico) è il parametro più
instabile e meno affidabile della teoria di Markowitz. I pesi
ottimali calcolati su un campione spesso non reggono fuori campione,
e talvolta perdono anche contro allocazioni banali (equal-weight,
60/40).

**Uso corretto della frontiera**: diagnostico (§3), non prescrittivo.
Se il PTF proposto è molto sotto la frontiera, è un segnale per
rivedere la diversificazione — non per copiare i pesi del punto
Max Sharpe.

La validazione statistica in §6 si applica quindi al **PTF reale
dell'utente** (`pf_proposed`), non ai pesi teorici della frontiera.

## §6 — Validazione statistica

Applicata al PTF reale dell'utente (pf_proposed), non ai pesi teorici
della frontiera. Stability pesi · MC Block A (confidence intervals) ·
MC Block B (skill ribilanciamento) · DSR · Decisione finale.

### §6.1 — Stability test pesi ottimali della frontiera (solo come misura di rumore)

Nota: questo test misura quanto è rumorosa la frontiera stessa
(quanto cambiano i suoi pesi Max Sharpe tra sotto-periodi) — NON
valida i pesi del mio PTF. Un CV alto qui conferma che il punto
Max Sharpe di §3 non va preso come riferimento di allocazione.

In [ ]:
stability = lazy_stability_weights(
    tickers=tickers,
    years=years,
    weight_bounds=(0, 1),
    n_splits=5,
    verbose=True,
)
if stability:
    print(f"\nRumorosità frontiera: {'bassa ✅' if stability['stable'] else 'alta ⚠️ (conferma: non adottare Max Sharpe come pesi)'}"
          f" (CV medio={stability['cv_mean']:.3f}, soglia={stability['cv_threshold']})")
else:
    print("⚠️  Stability test non eseguito — vedi messaggio diagnostico sopra.")

### §6.2 Monte Carlo 

Block A: Confidence Intervals sul PTF reale

Block B: Skill del ribilanciamento

In [ ]:
mc_report = run_mc_diagnostics(
    pf=pf_proposed,
    portfolio=portfolio,
    start_date=start_date,
    end_date=end_date,
    best_freq=best_freq,
    init_cash=init_cash,
    fees=fees,
    min_years=min_years
)

### §6.3 — DSR: Deflated Sharpe Ratio del PTF reale

Valida la significatività statistica dello Sharpe ratio di pf_proposed.
DSR > 0 → Sharpe statisticamente significativo dopo correzione
per multiple testing.

In [ ]:
_sr  = float(pf_proposed.sharpe_ratio())
_T   = int(pf_proposed.value().dropna().__len__())
_dsr = ofc_compute_dsr(sr_hat=_sr, n_trials=1, T=_T)
print(f"Sharpe ratio  : {_sr:.3f}")
print(f"T (obs)       : {_T}")
print(f"DSR           : {_dsr:.3f}  {'✅ significativo' if _dsr > 0 else '⚠️ non significativo'}")

### §6.4 — Verdetto finale

Valuta il PTF reale dell'utente (pesi e frequenza scelti in §1-§2),
non i pesi teorici della frontiera.

In [ ]:
# mc_report
mc_a2 = mc_report['a2']
mc_b  = mc_report['b']


cornice = max(65,len(portfolio_title)+10)
print("=" * cornice)
print(f" {BOLD}{portfolio_title}{RESET} — VERDETTO FINALE")
print("=" * cornice)

_checks = {
    'MC A2 Sharpe p50>0'  : mc_a2['percentiles']['p50']['Sharpe'] > 0,
    'MC B skill rebalance': mc_b['skill'],
    'DSR > 0'             : _dsr > 0,
}
_passed = sum(_checks.values())
for label, ok in _checks.items():
    print(f"  {'✅' if ok else '❌'}  {label}")
print("-" * cornice)
print(f"  Criteri superati: {_passed}/{len(_checks)}")
print(f"  Verdetto: {'✅ PROMOSSO' if _passed >= 2 else '❌ RIGETTATO'}")
print("=" * cornice)

## §7 — Relazione per l'investitore

In [ ]:
company_df = build_company_df_with_cache(["LU0034353002", "IE00BK5BQV03", "IE00BH04FZ00"])
print(company_df)
print(company_df.index)

In [ ]:
reports_dir = get_analysis_output_dir("l_analysis", ptf_name=f"{portfolio_title}_frequenza {best_freq}".replace(' ', '_').lower())
plots_dir   = reports_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
print(f"reports_dir: {reports_dir}")
print(f"plots_dir: {plots_dir}")

_ri_result = generate_relazione_investitore_report(
    out=out,
    portfolio=portfolio,   # un solo parametro, deriva tutto il resto
    reports_dir=reports_dir,
    year=year,
    # profile=profile,
)
